In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import scipy.io as sio
from dataclasses import dataclass
from typing import List, Tuple
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from natsort import natsorted
import numpy as np
import matplotlib.animation as animation
import xarray as xr
import h5py
import imageio
import matplotlib
import gc
import sys
import io
import matplotlib.colors as mcolors
import matplotlib.patches as patches
from scipy.optimize import curve_fit
import scipy.integrate
import re
import scipy.ndimage

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(rf'H:\codes\tidy3d'))

from AutomationModule import * 

import AutomationModule as AM
plt.rcParams.update({'font.size': 22})  

tidy3dAPI = os.environ["API_TIDY3D_KEY"]
plt.rc('font', family='Arial')


In [2]:
import time 
# time.sleep(50400)

In [3]:
file_data = "./data/L_1_12x/field_data_12x_raw_field_Ex_Ey_Ez_L_1_ff_0p2237_n_2p90_absorbers.h5"

data = AM.read_hdf5_as_dict(file_data)

Error reading HDF5 file: [Errno 2] Unable to synchronously open file (unable to open file: name = './data/L_1_12x/field_data_12x_raw_field_Ex_Ey_Ez_L_1_ff_0p2237_n_2p90_absorbers.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)


In [4]:
for path_direction in [
                  # rf"../../../../data/202606011_Beam_Spreading_12x_L_1_freqs_1700/n_2.90"
                  rf"../../../../data/20260602_Beam_Spreading_12x_L_1_freqs_1700_absorbers/n_2.90"
                       ]:
      folder_path = f"{path_direction}"
      
     
      for i,filename in enumerate(natsorted(os.listdir(folder_path))):
            lambda_range = re.search(r'lambda_(\d+(?:\.\d+)?)_([+-]?\d+(?:\.\d+)?)', filename)
            n = re.search(r'n_(\d+(?:\.\d+)?)', filename).group(1)
            if str(n) not in data.keys():
                  data[str(n)] = {}
            else:
                  print(filename)
                  print(n)
                  continue
            if os.path.isfile(os.path.join(folder_path, filename)):
                  file=os.path.join(folder_path, filename)
                  structure_1 = AM.loadFromFile(key = tidy3dAPI, file_path=file,get_ref=False)
                  field_data = structure_1.sim_data.monitor_data["monitorField_exit"]
                  data[str(n)] = {
                        "Ex":field_data.Ex.values,
                        "Ey":field_data.Ey.values,
                        "Ez":field_data.Ez.values,
                        "x":field_data.Ex.x.values,
                        "y":field_data.Ex.y.values,
                        "z":field_data.Ex.z.values,
                        "f":field_data.Ex.f.values
                  }

                 
                

Configured successfully.


13:33:41 W. Europe Daylight Time WARNING: Structure at 'structures[0]' has      
                                 bounds that extend exactly to simulation edges.
                                 This can cause unexpected behavior. If         
                                 intending to extend the structure to infinity  
                                 along one dimension, use td.inf as a size      
                                 variable instead to make this explicit.        

                                 WARNING: Suppressed 47 WARNING messages.       

14:26:39 W. Europe Daylight Time Billed flex credit cost: 70.689.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

In [5]:
os.makedirs("./data/L_1_12x",exist_ok=True)
AM.create_hdf5_from_dict(data,file_data)